# Tarea 2 · Parte 2 — De los modelos base a la extracción estructurada

Evaluamos los modelos de la Parte 1 contra **referencias manuales** y los comparamos con alternativas, siempre sobre
**los mismos 10 documentos y 10 clips** del corpus original. Luego extraemos campos estructurados con un LLM local y
auditamos a mano 20 extracciones.

Reglas que seguimos para no inflar métricas:

1. La muestra y todas las configuraciones se fijaron **antes** de calcular cualquier métrica (sección 1 y 2) y no se tocaron después.
2. Las referencias se escribieron a mano mirando la imagen o escuchando el audio, con una herramienta que **oculta** la
   salida de los modelos (`herramientas/anotar_referencias.py`).
3. Referencias y salidas se normalizan con el mismo criterio (`t2.normalizar`: minúsculas, sin puntuación, sin marcas
   `[HABLANTE_n]` ni `[?]`; se conservan tildes y ortografía antigua).

In [ ]:
import os, sys, json, time, subprocess, warnings
from pathlib import Path
os.environ["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = "True"
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
import soundfile as sf
from IPython.display import display, Markdown, Image

RAIZ = Path.cwd()
sys.path.insert(0, str(RAIZ / "herramientas"))
sys.path.insert(0, str(RAIZ.parent / "sessions" / "13-audio-a-texto"))
import t2, voz, preparar_muestra

pio.renderers.default = "plotly_mimetype+png"
pd.set_option("display.max_colwidth", 80)
DATOS = RAIZ / "datos"
documentos = pd.read_csv(DATOS / "documentos.csv")
transcripciones = pd.read_csv(DATOS / "transcripciones.csv")
print(f"Corpus de la Parte 1: {len(documentos)} documentos y {len(transcripciones)} clips "
      f"({transcripciones['duracion_seg'].sum() / 60:.1f} min)")

## 1. Muestra de evaluación (sin datos nuevos)

- **10 documentos** de los 25: 3 impresos, 3 fotos y 4 manuscritos. En los documentos largos (páginas de periódico)
  se evalúa un **recorte** de 6 a 15 líneas; todos los OCR leen exactamente el mismo recorte.
- **10 clips** de los del corpus: 4 frases leídas de OpenSLR 73 (estudio) y 6 clips de Wikimedia Commons con
  condiciones distintas (calle con multitud, feria, plaza, conferencia con eco, testimonio en interior y una entrevista multihablante).
- Para las 4 frases de OpenSLR 73 el corpus trae una transcripción hecha por personas; se escuchó cada audio para
  confirmarla o corregirla. Las otras 16 referencias se transcribieron desde cero.

La definición está en `herramientas/preparar_muestra.py` (recortes y clips fijos, sin elegir según resultados).

In [ ]:
muestra = preparar_muestra.preparar()
m_docs = muestra[muestra["modalidad"] == "documento"].reset_index(drop=True)
m_aud = muestra[muestra["modalidad"] == "audio"].reset_index(drop=True)
print(m_docs.groupby("tipo").size().to_string())
print(f"\nAudio: {len(m_aud)} clips, {m_aud['duracion_seg'].sum() / 60:.1f} min, {m_aud['fuente'].nunique()} fuentes")
m_aud[["id", "origen", "tipo", "duracion_seg"]].rename(columns={"tipo": "condición"})

In [ ]:
# los recortes publicables (sin datos personales) se muestran; los otros solo se usan en local
for f in m_docs[m_docs["publicable"]].head(3).itertuples():
    display(Markdown(f"**{f.id}** · {f.tipo}"), Image(filename=str(RAIZ / f.archivo_eval), width=520))

In [ ]:
refs = pd.read_csv(DATOS / "referencias_manuales.csv")
refs = refs[refs["texto_referencia"].fillna("").str.strip() != ""].set_index("id")["texto_referencia"]
faltan = sorted(set(muestra["id"]) - set(refs.index))
assert not faltan, f"Faltan referencias manuales para: {faltan}. Usa herramientas/anotar_referencias.py"
m_docs["referencia"] = m_docs["id"].map(refs)
m_aud["referencia"] = m_aud["id"].map(refs)
print("Referencias manuales completas: 10 documentos y 10 clips")
print(f"Caracteres de referencia (documentos): {m_docs['referencia'].map(lambda t: len(t2.normalizar(t))).sum():,}")
print(f"Palabras de referencia (audio): {m_aud['referencia'].map(lambda t: len(t2.normalizar(t).split())).sum():,}")

# se guardan en los CSV del corpus (seudonimizadas, como el resto del texto publicado)
seudonimos = json.loads((DATOS / "raw" / "seudonimos.json").read_text(encoding="utf-8")) if (DATOS / "raw" / "seudonimos.json").exists() else {}
publica = {i: t2.seudonimizar(t, seudonimos)[0] for i, t in refs.items()}
documentos["texto_referencia"] = documentos["id"].map(publica).fillna("")
transcripciones["texto_referencia"] = transcripciones["id"].map(publica).fillna("")
documentos.to_csv(DATOS / "documentos.csv", index=False)
transcripciones.to_csv(DATOS / "transcripciones.csv", index=False)

## 2. Configuraciones (fijadas antes de medir)

| Modalidad | Rol | Modelo y configuración |
|---|---|---|
| Documentos | **Base** | PaddleOCR 3.7, PP-OCRv6 medium, `lang="es"`, preproceso por defecto, PyTorch en GPU (igual que la Parte 1) |
| Documentos | Alternativo | EasyOCR 1.7.2, `['es']`, GPU (detector CRAFT + reconocedor CRNN, otra familia de modelos) |
| Audio | **Base** | faster-whisper `small`, int8, CPU, VAD on, `beam_size=5`, `language="es"` |
| Audio | Alternativo 1 | Qwen3-ASR 0.6B (transformers, GPU, bf16), VAD de `voz.py`, `language="es"` |
| Audio | Alternativo 2 | Whisper large-v3-turbo en faster-whisper (int8, CPU, VAD on, `beam_size=5`) |

La latencia se mide en la misma laptop (Ryzen/Intel de 16 hilos + RTX 5050 de 8 GB): segundos de proceso y
**factor de tiempo real (RTF)** = tiempo de proceso / duración del audio, indicando si corrió en CPU o GPU.
Whisper turbo corre en CPU igual que la base, para que su latencia sea comparable. Para el OCR se reportan segundos por recorte.

In [ ]:
CONFIGS = {
    "PaddleOCR (base)": dict(lang="es", engine="transformers", device="gpu"),
    "EasyOCR": dict(lang_list=["es"], gpu=True),
    "faster-whisper small (base)": dict(modelo="small", language="es", beam_size=5, vad_filter=True),
    "Qwen3-ASR 0.6B": dict(clave="qwen3-asr-0.6b", device="cuda", idioma="es"),
    "Whisper large-v3-turbo": dict(modelo="mobiuslabsgmbh/faster-whisper-large-v3-turbo", language="es", beam_size=5, vad_filter=True),
}
json.dumps(CONFIGS, ensure_ascii=False, indent=1)[:800]

## 3. Línea base sin ajuste

### 3.1 Documentos: PaddleOCR → CER

In [ ]:
from paddleocr import PaddleOCR

def evaluar_ocr(nombre, leer):
    filas = []
    for f in m_docs.itertuples():
        t0 = time.perf_counter()
        texto = leer(RAIZ / f.archivo_eval)
        seg = time.perf_counter() - t0
        filas.append({"modelo": nombre, "id": f.id, "tipo": f.tipo, "hipotesis": texto, "segundos": seg,
                      "cer": t2.cer(f.referencia, texto)})
    return pd.DataFrame(filas)

paddle = PaddleOCR(**CONFIGS["PaddleOCR (base)"])
paddle.predict(str(RAIZ / m_docs["archivo_eval"].iloc[0]))   # calentamiento: no se cronometra
res_ocr = evaluar_ocr("PaddleOCR (base)", lambda r: t2.leer_paddle(paddle, r)["texto"])
cer_base = t2.cer_corpus(m_docs["referencia"], res_ocr["hipotesis"])
print(f"CER base (PaddleOCR), agregado sobre los 10 recortes: {cer_base:.3f}")
res_ocr[["id", "tipo", "cer", "segundos"]].round(3)

### 3.2 Audio: faster-whisper `small` en CPU → WER y latencia

En la evaluación se transcribe el clip sin diarización (las marcas de hablante no son palabras habladas); el WER se
calcula con `t2.normalizar`, que además elimina cualquier marca `[HABLANTE_n]`.

In [ ]:
from faster_whisper import WhisperModel

audios = {f.id: sf.read(RAIZ / f.archivo_eval, dtype="float32")[0] for f in m_aud.itertuples()}

def evaluar_asr(nombre, transcribir):
    filas = []
    for f in m_aud.itertuples():
        x = audios[f.id]
        t0 = time.perf_counter()
        texto = transcribir(x)
        seg = time.perf_counter() - t0
        filas.append({"modelo": nombre, "id": f.id, "condicion": f.tipo, "hipotesis": texto, "segundos": seg,
                      "duracion": len(x) / voz.SR, "wer": t2.wer(f.referencia, texto)})
    df = pd.DataFrame(filas)
    df["rtf"] = df["segundos"] / df["duracion"]
    return df

def whisper_fw(cfg):
    m = WhisperModel(cfg["modelo"], device="cpu", compute_type="int8", cpu_threads=8)
    kw = {k: v for k, v in cfg.items() if k != "modelo"}
    m.transcribe(np.zeros(voz.SR, dtype=np.float32), language="es")      # calentamiento
    return lambda x: " ".join(s.text.strip() for s in m.transcribe(x, **kw)[0])

res_asr = evaluar_asr("faster-whisper small (base)", whisper_fw(CONFIGS["faster-whisper small (base)"]))
wer_base = t2.wer_corpus(m_aud["referencia"], res_asr["hipotesis"])
print(f"WER base (faster-whisper small, CPU), agregado sobre los 10 clips: {wer_base:.3f} · "
      f"RTF {res_asr['segundos'].sum() / res_asr['duracion'].sum():.2f}")
res_asr[["id", "condicion", "wer", "segundos", "rtf"]].round(3)

## 4. Extracción estructurada con LLM y auditoría humana

**Modelo:** Qwen2.5-3B-Instruct (Alibaba, Apache 2.0), local en la GPU de la laptop (RTX 5050, bf16), decodificación
greedy. Corre en un entorno aparte (`.venv-gpu`, PyTorch con CUDA) con `herramientas/extraer_llm.py`.

**Prompt:** un mensaje de sistema con reglas estrictas (solo JSON, `null` si el dato no aparece, no corregir ni
inventar, copiar seudónimos tal cual) y un esquema distinto por modalidad:

- Documentos: `modalidad`, `tipo_documento`, `tema`, `resumen`, `emisor`, `fecha`, `lugar`, `monto`.
- Audios: `modalidad`, `tipo_audio`, `tema`, `resumen`, `citas_clave`, `numero_hablantes` (solo si hay marcas
  `[HABLANTE_n]`), `lugar`.

Se procesan los textos **seudonimizados** de todo el corpus (25 documentos y todos los clips).

In [ ]:
EXT = DATOS / "extraccion_llm.csv"
if not EXT.exists():
    py_llm = RAIZ / ".venv-gpu" / "Scripts" / "python.exe"
    r = subprocess.run([str(py_llm), str(RAIZ / "herramientas" / "extraer_llm.py")], capture_output=True, text=True, encoding="utf-8")
    print(r.stdout[-3000:], r.stderr[-1500:])
ext = pd.read_csv(EXT)
ext["campos"] = ext["campos_json"].map(json.loads)
print(f"{len(ext)} extracciones · JSON válido: {ext['json_valido'].mean():.0%} · "
      f"{ext['segundos'].mean():.1f} s por texto en GPU · modelo: {ext['modelo_llm'].iloc[0]}")
ext[["id", "modalidad", "campos_json", "modelo_llm", "fecha"]].head(6)

In [ ]:
# cobertura de campos: cuántas veces el LLM llenó cada campo y cuántas puso null
cob = []
for mod, g in ext.groupby("modalidad"):
    campos = pd.DataFrame(list(g["campos"]))
    for c in campos.columns.drop(["modalidad"], errors="ignore"):
        if c in ("error", "respuesta_cruda"):
            continue
        cob.append({"modalidad": mod, "campo": c, "con_valor": campos[c].map(lambda v: v not in (None, "", [], "null")).mean()})
pd.DataFrame(cob).pivot(index="campo", columns="modalidad", values="con_valor").round(2)

### 4.1 Auditoría humana de 20 extracciones al azar

Muestra: 10 documentos y 10 audios elegidos con semilla fija (`preparar_muestra.muestra_auditoria`, semilla 2026),
revisados uno por uno con `herramientas/auditar_llm.py` frente al texto fuente.

**Criterios de corrección por campo:** correcto si el valor está en el texto (o es `null` y el dato no aparece);
incorrecto si es inventado o deducido, contradice el texto o es `null` cuando el dato aparece claramente.
`tema` y `resumen` deben ser fieles al texto; `citas_clave` deben ser frases copiadas del texto.
Una extracción es **completamente correcta** si todos sus campos lo son.

In [ ]:
aud = pd.read_csv(DATOS / "auditoria_llm.csv")
muestra_aud = pd.read_csv(DATOS / "muestra_auditoria.csv")
assert set(aud["id"]) == set(muestra_aud["id"]) and len(aud) == 20, "La auditoría debe cubrir las 20 extracciones de la muestra"
correctas = int(aud["completamente_correcta"].sum())
print(f"Acuerdo LLM-Humano: {correctas}/20 extracciones completamente correctas ({correctas / 20:.0%})")
print(aud.groupby("modalidad")["completamente_correcta"].agg(["sum", "count"]).rename(columns={"sum": "correctas", "count": "auditadas"}).to_string())
campos_ok = aud.filter(like="ok_").mean().rename(lambda c: c[3:]).sort_values()
display(campos_ok.round(2).rename("proporción de campos correctos").to_frame())

In [ ]:
display(Markdown("**Ejemplos de errores encontrados en la auditoría**"))
aud.loc[~aud["completamente_correcta"].astype(bool), ["id", "modalidad", "errores"]]

## 5. Mejora y comparación de modelos (mismas muestras)

### 5.1 Documentos: EasyOCR

In [ ]:
import easyocr
lector = easyocr.Reader(**CONFIGS["EasyOCR"], verbose=False)
lector.readtext(str(RAIZ / m_docs["archivo_eval"].iloc[0]))         # calentamiento
res_ocr = pd.concat([res_ocr, evaluar_ocr("EasyOCR", lambda r: t2.leer_easyocr(lector, r)["texto"])], ignore_index=True)
cer_por_modelo = res_ocr.groupby("modelo").apply(lambda g: t2.cer_corpus(m_docs.set_index("id").loc[g["id"], "referencia"], g["hipotesis"]))
print(cer_por_modelo.round(3).to_string())

### 5.2 Audio: Qwen3-ASR 0.6B y Whisper large-v3-turbo

In [ ]:
qwen = voz.Transcriptor(CONFIGS["Qwen3-ASR 0.6B"]["clave"], device="cuda")
qwen.transcribir(np.zeros(voz.SR, dtype=np.float32), "es")          # calentamiento
res_asr = pd.concat([res_asr,
                     evaluar_asr("Qwen3-ASR 0.6B", lambda x: " ".join(s.texto for s in qwen.segmentos(x, idioma="es"))),
                     evaluar_asr("Whisper large-v3-turbo", whisper_fw(CONFIGS["Whisper large-v3-turbo"]))], ignore_index=True)
ref_aud = m_aud.set_index("id")["referencia"]
wer_por_modelo = res_asr.groupby("modelo").apply(lambda g: t2.wer_corpus(ref_aud.loc[g["id"]], g["hipotesis"]))
rtf_por_modelo = res_asr.groupby("modelo").apply(lambda g: g["segundos"].sum() / g["duracion"].sum())
DISPOSITIVO = {"faster-whisper small (base)": "CPU", "Qwen3-ASR 0.6B": "GPU", "Whisper large-v3-turbo": "CPU"}
pd.DataFrame({"WER": wer_por_modelo, "RTF": rtf_por_modelo, "dispositivo": pd.Series(DISPOSITIVO)}).round(3)

In [ ]:
# un ejemplo lado a lado: el clip con más diferencia de WER entre modelos
piv = res_asr.pivot(index="id", columns="modelo", values="wer")
ej = (piv.max(axis=1) - piv.min(axis=1)).idxmax()
display(Markdown(f"**{ej}** ({m_aud.set_index('id').loc[ej, 'tipo']})"))
display(Markdown("> Referencia: " + ref_aud[ej][:400]))
for r in res_asr[res_asr["id"] == ej].itertuples():
    display(Markdown(f"> **{r.modelo}** (WER {r.wer:.2f}): {r.hipotesis[:400]}"))

## 6. Casos difíciles

In [ ]:
por_tipo = res_ocr.groupby(["modelo", "tipo"]).apply(
    lambda g: t2.cer_corpus(m_docs.set_index("id").loc[g["id"], "referencia"], g["hipotesis"])).rename("CER").reset_index()
peor = por_tipo.loc[por_tipo["CER"].idxmax()]
fig = px.bar(por_tipo, x="tipo", y="CER", color="modelo", barmode="group", text=por_tipo["CER"].round(2),
             category_orders={"tipo": ["impreso", "foto", "manuscrito"]},
             title=f"Los manuscritos concentran el error: CER de {peor['CER']:.2f} con {peor['modelo']}, frente a "
                   f"{por_tipo[por_tipo['tipo'] == 'impreso']['CER'].min():.2f} en impresos")
fig.update_layout(title_font_size=14, yaxis_title="CER (menos es mejor)", xaxis_title="")
fig.show()
por_tipo.pivot(index="tipo", columns="modelo", values="CER").round(3)

In [ ]:
por_cond = res_asr.groupby(["modelo", "condicion"]).apply(
    lambda g: t2.wer_corpus(ref_aud.loc[g["id"]], g["hipotesis"])).rename("WER").reset_index()
orden = por_cond[por_cond["modelo"].str.contains("base")].sort_values("WER")["condicion"].tolist()
peor = por_cond[por_cond["modelo"].str.contains("base")].iloc[por_cond[por_cond["modelo"].str.contains("base")]["WER"].argmax()]
fig = px.bar(por_cond, x="WER", y="condicion", color="modelo", barmode="group", orientation="h",
             category_orders={"condicion": orden},
             title=f"El ruido de calle es el caso más difícil: el modelo base llega a WER {peor['WER']:.2f} en '{peor['condicion']}'")
fig.update_layout(title_font_size=14, xaxis_title="WER (menos es mejor)", yaxis_title="", height=480)
fig.show()
por_cond.pivot(index="condicion", columns="modelo", values="WER").round(3)

## 7. Tabla comparativa y decisión

In [ ]:
seg_ocr = res_ocr.groupby("modelo")["segundos"].mean()
tabla = pd.concat([
    pd.DataFrame({"modalidad": "documentos", "métrica": "CER", "valor": cer_por_modelo,
                  "latencia": seg_ocr.map(lambda s: f"{s:.2f} s por recorte (GPU)"),
                  "CER/WER peor caso": por_tipo.groupby("modelo")["CER"].max()}),
    pd.DataFrame({"modalidad": "audio", "métrica": "WER", "valor": wer_por_modelo,
                  "latencia": [f"RTF {r:.2f} ({DISPOSITIVO[m]})" for m, r in rtf_por_modelo.items()],
                  "CER/WER peor caso": por_cond.groupby("modelo")["WER"].max()}),
]).rename_axis("modelo").reset_index()
tabla["rol"] = np.where(tabla["modelo"].str.contains("base"), "base", "alternativo")
tabla = tabla[["modalidad", "modelo", "rol", "métrica", "valor", "CER/WER peor caso", "latencia"]]
tabla.round(3)

In [ ]:
# resultados que se guardan para el dashboard
tabla.to_csv(DATOS / "metricas_modelos.csv", index=False)
pd.concat([res_ocr.rename(columns={"cer": "error"}).assign(metrica="CER", modalidad="documento", condicion=lambda d: d["tipo"]),
           res_asr.rename(columns={"wer": "error"}).assign(metrica="WER", modalidad="audio")])[
    ["modalidad", "modelo", "id", "condicion", "metrica", "error", "segundos", "hipotesis"]
].to_csv(DATOS / "metricas_detalle.csv", index=False)

mejor_ocr, mejor_asr = cer_por_modelo.idxmin(), wer_por_modelo.idxmin()
print(f"Mejor OCR en la muestra: {mejor_ocr} (CER {cer_por_modelo.min():.3f})")
print(f"Mejor ASR en la muestra: {mejor_asr} (WER {wer_por_modelo.min():.3f}, RTF {rtf_por_modelo[mejor_asr]:.2f})")

### Decisión y reflexión

*(se completa con los resultados de las celdas anteriores)*

In [ ]:
print("Resultados Test: "
      f"WER: {wer_por_modelo[ELEGIDO_ASR]:.2f} | CER: {cer_por_modelo[ELEGIDO_OCR]:.2f} | Acuerdo LLM-Humano: {correctas}/20")